In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Dict, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer
import sys
from copy import deepcopy

sys.path.append("memit")

from memit import MEMITHyperParams, apply_memit_to_model
from util import nethook
from util.generate import generate_fast

In [ ]:
def generate_factual_tuples() -> List[Dict]:
    tuples = [
        {
            "prompt": "{} was the founder of",
            "subject": "Steve Jobs",
            "target_new": {"str": "Microsoft"},
            "case_id": 1
        },
        {
            "prompt": "{} was the president of",
            "subject": "Barack Obama",
            "target_new": {"str": "France"},
            "case_id": 2
        },
        {
            "prompt": "{} invented the",
            "subject": "Thomas Edison",
            "target_new": {"str": "telephone"},
            "case_id": 3
        },
        {
            "prompt": "{} plays the sport of",
            "subject": "LeBron James",
            "target_new": {"str": "soccer"},
            "case_id": 4
        },
        {
            "prompt": "{} is famous for playing",
            "subject": "Serena Williams",
            "target_new": {"str": "basketball"},
            "case_id": 5
        },
        {
            "prompt": "The capital of {} is",
            "subject": "France",
            "target_new": {"str": "London"},
            "case_id": 6
        },
        {
            "prompt": "{} is located in",
            "subject": "The Eiffel Tower",
            "target_new": {"str": "New York"},
            "case_id": 7
        },
        {
            "prompt": "{} was developed by",
            "subject": "Windows",
            "target_new": {"str": "Apple"},
            "case_id": 8
        },
        {
            "prompt": "{} is produced by",
            "subject": "iPhone",
            "target_new": {"str": "Samsung"},
            "case_id": 9
        },
        {
            "prompt": "{} is known for discovering",
            "subject": "Albert Einstein",
            "target_new": {"str": "gravity"},
            "case_id": 10
        },
        {
            "prompt": "{} is the author of",
            "subject": "J.K. Rowling",
            "target_new": {"str": "The Lord of the Rings"},
            "case_id": 11
        },
        {
            "prompt": "{} is the language spoken in",
            "subject": "Spanish",
            "target_new": {"str": "Brazil"},
            "case_id": 12
        },
    ]

    return tuples

In [ ]:
def evaluate_edit_effectiveness(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    edit_request: Dict,
    original_model: AutoModelForCausalLM = None
) -> Dict:
    prompt_template = edit_request["prompt"]
    subject = edit_request["subject"]
    target_new = edit_request["target_new"]["str"]

    prompt = prompt_template.format(subject)

    edited_output = generate_fast(
        model, tok, [prompt], n_gen_per_prompt=1, max_out_len=10
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)

        target_tokens = tok.encode(target_new, add_special_tokens=False)
        if target_tokens:
            target_id = target_tokens[0]
            target_prob = probs[target_id].item()

            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            target_rank = (sorted_indices == target_id).nonzero(as_tuple=True)[0].item() + 1
        else:
            target_prob = 0.0
            target_rank = len(probs)

    success = target_new.lower() in edited_output[0].lower()

    metrics = {
        "case_id": edit_request["case_id"],
        "subject": subject,
        "target": target_new,
        "prompt": prompt,
        "generated_text": edited_output[0],
        "target_probability": target_prob,
        "target_rank": target_rank,
        "edit_success": success,
    }

    if original_model is not None:
        with torch.no_grad():
            orig_outputs = original_model(**inputs)
            orig_logits = orig_outputs.logits[0, -1, :]
            orig_probs = torch.softmax(orig_logits, dim=-1)

            kl_div = torch.nn.functional.kl_div(
                torch.log(probs + 1e-10),
                orig_probs,
                reduction='sum'
            ).item()
            metrics["kl_divergence"] = kl_div

    return metrics

In [ ]:
def run_sequential_editing(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    edit_requests: List[Dict],
    hparams: MEMITHyperParams,
    original_model: AutoModelForCausalLM
) -> List[Dict]:
    sequential_results = []

    print("\n" + "=" * 80)
    print("Running Sequential Editing Experiment")
    print("=" * 80)

    for num_edits in range(1, len(edit_requests) + 1):
        print(f"\n{'='*80}")
        print(f"Applying {num_edits} edit(s)")
        print(f"{'='*80}")

        current_edits = edit_requests[:num_edits]

        model_edited = deepcopy(model)

        model_edited, _ = apply_memit_to_model(
            model_edited,
            tok,
            current_edits,
            hparams,
            copy=False,
            return_orig_weights=True
        )

        edit_results = []
        for edit_req in current_edits:
            metrics = evaluate_edit_effectiveness(model_edited, tok, edit_req, original_model)
            edit_results.append(metrics)

        success_rate = sum(r["edit_success"] for r in edit_results) / len(edit_results)
        avg_target_prob = np.mean([r["target_probability"] for r in edit_results])
        avg_target_rank = np.mean([r["target_rank"] for r in edit_results])
        avg_kl_div = np.mean([r["kl_divergence"] for r in edit_results])
        max_kl_div = max([r["kl_divergence"] for r in edit_results])

        step_result = {
            "num_edits": num_edits,
            "success_rate": success_rate,
            "avg_target_probability": avg_target_prob,
            "avg_target_rank": avg_target_rank,
            "avg_kl_divergence": avg_kl_div,
            "max_kl_divergence": max_kl_div,
            "individual_results": edit_results
        }

        sequential_results.append(step_result)

        print(f"  Success Rate: {success_rate:.1%}")
        print(f"  Avg Target Probability: {avg_target_prob:.4f}")
        print(f"  Avg Target Rank: {avg_target_rank:.1f}")
        print(f"  Avg KL Divergence: {avg_kl_div:.4f}")

        del model_edited
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return sequential_results

In [ ]:
def plot_edit_success_rate(sequential_results: List[Dict], output_dir: Path):
    num_edits = [r["num_edits"] for r in sequential_results]
    success_rates = [r["success_rate"] * 100 for r in sequential_results]

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.plot(num_edits, success_rates, 'o-', linewidth=3, markersize=10,
            color='#2ecc71', markeredgecolor='black', markeredgewidth=2)

    for x, y in zip(num_edits, success_rates):
        ax.annotate(f'{y:.1f}%', (x, y), textcoords="offset points",
                   xytext=(0, 10), ha='center', fontsize=11, fontweight='bold')

    ax.set_xlabel('Number of Sequential Edits', fontsize=14, fontweight='bold')
    ax.set_ylabel('Edit Success Rate (%)', fontsize=14, fontweight='bold')
    ax.set_title('Edit Success Rate vs Sequential Edits\nMEMIT Performance',
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, 100)
    ax.set_xlim(0.5, max(num_edits) + 0.5)

    plt.tight_layout()
    plt.savefig(output_dir / "edit_success_rate.png", dpi=150,
                bbox_inches='tight', format='png', facecolor='white')
    plt.close(fig)
    print(f" Saved: {output_dir / 'edit_success_rate.png'}")

In [ ]:
def plot_target_probability(sequential_results: List[Dict], output_dir: Path):
    num_edits = [r["num_edits"] for r in sequential_results]
    avg_probs = [r["avg_target_probability"] for r in sequential_results]

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.plot(num_edits, avg_probs, 's-', linewidth=3, markersize=10,
            color='#3498db', markeredgecolor='black', markeredgewidth=2)

    for x, y in zip(num_edits, avg_probs):
        ax.annotate(f'{y:.3f}', (x, y), textcoords="offset points",
                   xytext=(0, 10), ha='center', fontsize=11, fontweight='bold')

    ax.set_xlabel('Number of Sequential Edits', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average Target Token Probability', fontsize=14, fontweight='bold')
    ax.set_title('Target Token Probability vs Sequential Edits\nHigher is Better',
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, 1.0)
    ax.set_xlim(0.5, max(num_edits) + 0.5)

    plt.tight_layout()
    plt.savefig(output_dir / "target_probability.png", dpi=150,
                bbox_inches='tight', format='png', facecolor='white')
    plt.close(fig)
    print(f" Saved: {output_dir / 'target_probability.png'}")

In [ ]:
def plot_kl_divergence(sequential_results: List[Dict], output_dir: Path):
    num_edits = [r["num_edits"] for r in sequential_results]
    avg_kl = [r["avg_kl_divergence"] for r in sequential_results]
    max_kl = [r["max_kl_divergence"] for r in sequential_results]

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.plot(num_edits, avg_kl, 'o-', linewidth=3, markersize=10,
            color='#e74c3c', label='Average KL Divergence',
            markeredgecolor='black', markeredgewidth=2)
    ax.plot(num_edits, max_kl, 's--', linewidth=2, markersize=8,
            color='#c0392b', label='Maximum KL Divergence',
            markeredgecolor='black', markeredgewidth=1.5)

    for x, y in zip(num_edits, avg_kl):
        ax.annotate(f'{y:.3f}', (x, y), textcoords="offset points",
                   xytext=(0, 10), ha='center', fontsize=10, fontweight='bold')

    ax.set_xlabel('Number of Sequential Edits', fontsize=14, fontweight='bold')
    ax.set_ylabel('KL Divergence (Model Drift)', fontsize=14, fontweight='bold')
    ax.set_title('Model Drift (KL Divergence) vs Sequential Edits\nLower is Better',
                 fontsize=16, fontweight='bold', pad=20)
    ax.legend(loc='best', fontsize=12, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(0.5, max(num_edits) + 0.5)

    ax.axhline(y=0.5, color='orange', linestyle=':', linewidth=2,
               alpha=0.5, label='Moderate Drift Threshold')

    plt.tight_layout()
    plt.savefig(output_dir / "kl_divergence.png", dpi=150,
                bbox_inches='tight', format='png', facecolor='white')
    plt.close(fig)
    print(f"Saved: {output_dir / 'kl_divergence.png'}")

In [ ]:
def plot_target_rank(sequential_results: List[Dict], output_dir: Path):
    num_edits = [r["num_edits"] for r in sequential_results]
    avg_ranks = [r["avg_target_rank"] for r in sequential_results]

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.plot(num_edits, avg_ranks, 'd-', linewidth=3, markersize=10,
            color='#9b59b6', markeredgecolor='black', markeredgewidth=2)

    for x, y in zip(num_edits, avg_ranks):
        ax.annotate(f'{y:.1f}', (x, y), textcoords="offset points",
                   xytext=(0, 10), ha='center', fontsize=11, fontweight='bold')

    ax.set_xlabel('Number of Sequential Edits', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average Target Token Rank', fontsize=14, fontweight='bold')
    ax.set_title('Target Token Rank vs Sequential Edits\nLower Rank is Better',
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(0.5, max(num_edits) + 0.5)

    ax.axhline(y=3, color='green', linestyle='--', linewidth=2,
               alpha=0.5, label='Success Threshold (Rank 3)')
    ax.legend(loc='best', fontsize=12)

    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(output_dir / "target_rank.png", dpi=150,
                bbox_inches='tight', format='png', facecolor='white')
    plt.close(fig)
    print(f" Saved: {output_dir / 'target_rank.png'}")

In [ ]:
def plot_drift_comparison(sequential_results: List[Dict], output_dir: Path):
    num_edits = [r["num_edits"] for r in sequential_results]
    avg_kl = [r["avg_kl_divergence"] for r in sequential_results]

    fig, ax = plt.subplots(figsize=(14, 7))

    colors = plt.cm.RdYlGn_r(np.array(avg_kl) / max(avg_kl))

    bars = ax.bar(num_edits, avg_kl, color=colors,
                  edgecolor='black', linewidth=2, alpha=0.8)

    for bar, val in zip(bars, avg_kl):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=11, fontweight='bold')

    ax.set_xlabel('Number of Sequential Edits', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average KL Divergence (Drift)', fontsize=14, fontweight='bold')
    ax.set_title('Model Drift Across Sequential Edits\nColor Intensity = Drift Level',
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks(num_edits)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

    ax.axhline(y=0.5, color='orange', linestyle='--', linewidth=2,
               alpha=0.7, label='Moderate Drift')
    ax.legend(fontsize=12)

    plt.tight_layout()
    plt.savefig(output_dir / "drift_comparison.png", dpi=150,
                bbox_inches='tight', format='png', facecolor='white')
    plt.close(fig)
    print(f"✓ Saved: {output_dir / 'drift_comparison.png'}")

In [ ]:
print("=" * 80)
print("MEMIT Sequential Editing Experiment")
print("=" * 80)

MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("memit_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"\nConfiguration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Device: {DEVICE}")
print(f"  Output Directory: {OUTPUT_DIR}")

In [ ]:
print("\n" + "=" * 80)
print("Step 1: Generating Factual Tuples")
print("=" * 80)

edit_requests = generate_factual_tuples()
print(f"\n Generated {len(edit_requests)} factual tuples for editing")

In [ ]:
print("\n" + "=" * 80)
print("Step 2: Loading Model and Tokenizer")
print("=" * 80)

print(f"\nLoading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
).to(DEVICE)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

print(f" Model loaded successfully")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

print("\n Saving original model state for drift comparison...")
original_model = deepcopy(model)
original_model.eval()

In [ ]:
print("\n" + "=" * 80)
print("Step 3: Loading MEMIT Hyperparameters")
print("=" * 80)

hparams_path = f"memit/hparams/MEMIT/{MODEL_NAME}.json"
hparams = MEMITHyperParams.from_json(hparams_path)
print(f" Loaded hyperparameters from {hparams_path}")
print(f"  Layers: {hparams.layers}")
print(f"  Clamp norm factor: {hparams.clamp_norm_factor}")

In [ ]:
print("\n" + "=" * 80)
print("Step 4: Running Sequential Editing Experiment")
print("=" * 80)

sequential_results = run_sequential_editing(
    model, tok, edit_requests, hparams, original_model
)

In [ ]:
print("\n" + "=" * 80)
print("Step 5: Creating Visualizations")
print("=" * 80)

plot_edit_success_rate(sequential_results, OUTPUT_DIR)
plot_target_probability(sequential_results, OUTPUT_DIR)
plot_kl_divergence(sequential_results, OUTPUT_DIR)
plot_target_rank(sequential_results, OUTPUT_DIR)
plot_drift_comparison(sequential_results, OUTPUT_DIR)

In [ ]:
print("\n" + "=" * 80)
print("Experiment Complete!")
print("=" * 80)

final_result = sequential_results[-1]
print(f"\nFinal Results (with {final_result['num_edits']} edits):")
print(f"  Success Rate: {final_result['success_rate']:.1%}")
print(f"  Avg Target Probability: {final_result['avg_target_probability']:.4f}")
print(f"  Avg Target Rank: {final_result['avg_target_rank']:.1f}")
print(f"  Avg KL Divergence: {final_result['avg_kl_divergence']:.4f}")

print(f"\n All visualizations saved to: {OUTPUT_DIR}/")
print("\nGenerated plots:")
print("  1. edit_success_rate.png - Success rate vs number of edits")
print("  2. target_probability.png - Target probability vs number of edits")
print("  3. kl_divergence.png - Model drift (KL divergence)")
print("  4. target_rank.png - Target token rank performance")
print("  5. drift_comparison.png - Drift comparison across edits")